In [ ]:
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
import gzip

from qiskit.quantum_info import SparsePauliOp

from matplotlib import pyplot as plt

from qphaset.phases import gstates_to_rdms_matrix_qs_mps, phases_vfield
from qphaset.plotting import plot_grad_g_angle_stream
from qphaset.linalg import schmidt_decomp_half, schmidt_decomp_2q_angles

## General solution (beta) for the order parameter discovery.
This is the (beta!) implementation of the most general form of the order parameter discovery (in the paper, this is the second problem in the section deicated to the order parameter discovery, soon more details will be expanded).

In [ ]:
# *** Recipes ***

# TODO align variables naming with paper.

filename = "annni_ext-02e131bf-8a7e-4ad4-845a-dcc271c34661.pkl"
filename = "results/data/ANNNI-8e2b8bce-1d27-41d0-a48b-fbce8d4ab8cc.pkl" # ANNNI 50, c1=-1, upside down, 64 x 64
eta = 0
gamma = 10
m_ev_idx = 0
obs_ev_idx = 1
v0_first_schmidt_vec = False

# filename = "annni_ext-02e131bf-8a7e-4ad4-845a-dcc271c34661.pkl"
# eta = np.pi
# gamma = 1
# m_ev_idx = 2 # (suboptimal solution)
# obs_ev_idx = 3
# v0_first_schmidt_vec = False

# filename = "annni_ext-02e131bf-8a7e-4ad4-845a-dcc271c34661.pkl"
# eta = 0
# gamma = 1
# m_ev_idx = 1 # (suboptimal solution)
# obs_ev_idx = 3 # also 2 with v0_first_schmidt_vec=False
# v0_first_schmidt_vec = True
# Try shifting sites eg sites = [l // 2 + 1], note that the same concept selected by obs_ev_idx may swap position
# due to numerical errors.

# Other data sources to be considered:
# filename = "annni_ext-acea288f-fb29-4c73-8438-4e44ce8ab467.pkl"
# filename = "annni_ext-bcaf303a-925e-48f4-a944-9eb777dcfc32.pkl"

In [ ]:
# Data loading

with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

sites = [l // 2, (l // 2) + 1]

rdms = gstates_to_rdms_matrix_qs_mps(gstates, sites=sites)
rdms = rdms[::-1]   # TODO fix

In [ ]:
rdms.shape

In [ ]:
# eta = 0  # Adjust st phases have opposite signs.
# Most of the times eta=0 is good, however, use eta=pi to obtain the complementary
# order parameter. 

### Choose subset of rdms to bias the order parameter

In [ ]:
# params_zoom = params.reshape((64,64,2))[:32,:32].reshape((32*32,2)) # we take (k=(0.0,0.5), h=(1.0,0.5))
# params_zoom = params.reshape((64,64,2))[32:,32:].reshape((32*32,2)) # we take (k=(0.5,1.0), h=(0.0,0.5))
params_zoom = params.reshape((64,64,2))[:32,32:].reshape((32*32,2)) # we take (k=(0.5,1.0), h=(0.5,1.0))
params_extent_zoom = np.concatenate([np.min(params_zoom, axis=0), np.max(params_zoom, axis=0)])
params_extent_zoom = tuple(params_extent_zoom[[0, 2, 1, 3]])
params_extent_zoom

In [ ]:
# Ising transition parameters
gstate_zoom = np.asarray(gstates).reshape((64,64))[32:,32:].reshape((32*32))
print(gstate_zoom.shape)
rdms_zoom = gstates_to_rdms_matrix_qs_mps(gstate_zoom, sites=sites)
rdms_zoom = rdms_zoom[::-1] 

In [ ]:
grad_g = phases_vfield(rdms_zoom, scale=1)
eta = np.pi
ys = np.sin(np.angle(grad_g) + eta)

# Labels plot
plt.matshow(np.sign(ys), origin='lower', extent=params_extent_zoom)
plt.colorbar()

In [ ]:
rdms_zoom = rdms_zoom[1:-1, 1:-1] # TODO fix

In [ ]:
lattice_shape = rdms_zoom.shape[:2]
rdms_zoom = np.reshape(rdms_zoom, (-1, ) + rdms_zoom.shape[2:])
ys = ys.flatten()

### Solve equation $B5$ in the appendix of the manuscript

In [ ]:
assert rdms_zoom.ndim == 3


def mats_to_vv_sum(mats):
    mats = np.asarray(mats)
    # Vectorize each matrix (row-major)
    mats = np.reshape(mats, (len(mats), -1))
    # vec(rho)vec(rho)^H
    mats = np.einsum('ia, ib -> iab', mats, np.conj(mats))
    return np.sum(mats, axis=0)


# Normalization factors.
fp, fn = 1 / np.count_nonzero(ys > 0), 1 / np.count_nonzero(ys < 0)

# Tradeoff parameter.
# gamma = 10

pos_y_term = -fp * mats_to_vv_sum(rdms_zoom[np.nonzero(ys > 0)])
neg_y_term = fn * mats_to_vv_sum(rdms_zoom[np.nonzero(ys < 0)])
m = pos_y_term + gamma * neg_y_term ### Eq. B5
# Check that matrix m is numerically close to Hermitian.
assert np.allclose(m - m.conj(m).T, 0)

m_eigval, m_eigvec = np.linalg.eigh(m)
# Eigenvector selection, default 0.
# m_ev_idx = 0     # Use value greater than 0 for sub-optimal but interesting solutions (eg floating phase).
# Observable is taken as the eigenvector (reshaped to matrix) corresponding to the least eigenvalue (when eigvidx = 0,
# if there is multiplicity for now we ignore it) as it becomes 0 when the matrix m is shifted by -I * lambda_min (solution of the dual problem).
obs = np.reshape(m_eigvec[:, m_ev_idx:(m_ev_idx + 1)], rdms_zoom[0].shape)

# TODO *** Consider multi dimensional eigenspaces ***.

# Check the observable is numerically close to Hermitian.
assert np.allclose(obs - np.conj(obs).T, 0)

In [ ]:
# Sanity-check. 0Check that candidate observables not numerically close to Hermitian
# correspond to eigenvalues numerically close to 0.

for i in range(len(m_eigvec)):
    test_obs = np.reshape(m_eigvec[:, i:(i + 1)], rdms[0].shape)
    if not np.allclose(test_obs - np.conj(test_obs).T, 0):
        assert np.isclose(m_eigval[i], 0)

In [ ]:
# TODO Sanity-check on the presence of positive and negative eigenvalues not numerically close to 0.
# Note these are not the eigenvalues of the observable.
m_eigval

In [ ]:
obs = obs.real

In [ ]:
plt.matshow(obs)
plt.colorbar()

In [ ]:
obs_eval, obs_ev = np.linalg.eigh(obs)
# Eigenvalues of the observable, check here the magnitudes (check also the sign!) and explore the specific projectors
# by setting the variable obs_ev_idx with the index of the selected eigenvector.
obs_eval

In [ ]:
SparsePauliOp.from_operator(obs)

In [ ]:
obs

In [ ]:
lattice_shape = rdms.shape[:2]
rdms = np.reshape(rdms, (-1, ) + rdms.shape[2:])
ys = ys.flatten()

meas = [np.trace(rdm @ obs) for rdm in rdms]
meas = np.reshape(meas, lattice_shape)

# Note we plot the **absolute value**. The ordered phase can assume any non-zero value (positive and negative).
# So, we identify the disordered phase with the values close to 0 (numerically).
plt.matshow(np.abs(meas), origin='lower', cmap='plasma', extent=params_extent)
plt.colorbar()

In [ ]:
# TODO Operator-Schmidt decomposition of obs to reveal if we are numerically close to product operator
# (useful for the floating phase).

In [ ]:
# ************************************************************
# *** Analysis of a selected eigenvector of the observable ***
# ************************************************************
# Key parameters: obs_ev_idx (eigenvector index) and v0_first_schmidt_vec.

In [ ]:
# Select a rank-1 observable (note the ordering of the eigenvalues of obs).
# obs_ev_idx = 2

v0 = obs_ev[:, 2]

# Schmidt coefficients w.r.t. middle split
v0_schmidt_coeffs = schmidt_decomp_half(v0)[1]
v0_schmidt_coeffs

In [ ]:
# Angles for each qubit in the decomposition of v0 (assumed 2 qubits state).
schmidt_decomp_2q_angles(v0)[0]

In [ ]:
# Optionally take first component of Schmidt decomp.
# v0_first_schmidt_vec = True
obs_ev_idx = 2

if v0_first_schmidt_vec:
    v0 = schmidt_decomp_half(v0, contract_sigmas=1, normalize=True)

v0 = np.reshape(v0, (-1, 1))
obs0 = np.sign(obs_eval[obs_ev_idx]) * v0 @ np.conj(v0.T)

SparsePauliOp.from_operator(obs0)

In [ ]:
plt.matshow(obs0.real)
plt.colorbar()

In [ ]:
meas = [np.trace(rdm @ obs0) for rdm in rdms]
meas = np.reshape(meas, lattice_shape)

# Note we plot the absolute value to avoid misunderstanding in the interpretation of the
# colormap.
plt.matshow(np.abs(meas), origin='lower', cmap='plasma', extent=params_extent)
plt.xlabel('$\\kappa$')
plt.ylabel('h')
plt.colorbar()

In [ ]:
obs0

In [ ]:
plot_grad_g_angle_stream(grad_g, params_extent=params_extent, theory_lines=False);